# Plant classifier — CNN training (PyTorch)

This notebook fine-tunes a pretrained ResNet-18 on your 10 plant folders. Before running it, select **Runtime → Change runtime type → T4 GPU** if available. Free GPU availability can vary.

Keep the images in Google Drive, separate from GitHub. Expected Drive path: `MyDrive/plant_classifier/images/<plant folder>/<photo>.jpeg`. The folder names become the class labels.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select a GPU runtime from Runtime → Change runtime type.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Connect your GitHub project

After you create the GitHub repository and upload the project files, paste its URL below. The repository should contain `train_cnn.py` at its top level.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE'
PROJECT_DIR = Path('/content/plant_classifier')
if REPO_URL.startswith('PASTE_'):
    raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'train_cnn.py').is_file(), 'train_cnn.py was not found at the repository root.'
print('Project ready at:', PROJECT_DIR)

## Find and prepare the dataset

Upload the `images` folder to Google Drive at `MyDrive/plant_classifier/images`. This cell checks the class folders and copies the images to the temporary Colab disk for faster training.

In [ ]:
from collections import Counter
from pathlib import Path
import shutil

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/plant_classifier/images')
LOCAL_DATA_DIR = Path('/content/plant_images')
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
if not DRIVE_DATA_DIR.is_dir():
    raise FileNotFoundError(f'Upload the images folder to: {DRIVE_DATA_DIR}')
counts = {}
for folder in sorted(p for p in DRIVE_DATA_DIR.iterdir() if p.is_dir()):
    counts[folder.name] = sum(1 for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in extensions)
print('Class image counts:', counts)
if len(counts) != 10 or any(n == 0 for n in counts.values()):
    raise ValueError(f'Expected 10 non-empty plant folders, found {len(counts)}.')
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
print('Copied dataset to temporary runtime disk:', LOCAL_DATA_DIR)

## Install the small extra dependency and train

Colab already provides a PyTorch build for its runtime. Install only scikit-learn here so pip does not replace the GPU-enabled PyTorch installation. The best checkpoint and reports are written to Drive so they survive the session.

In [ ]:
%pip -q install scikit-learn
import sys
OUTPUT_DIR = Path('/content/drive/MyDrive/plant_classifier/outputs/cnn')
subprocess.run([
    sys.executable, str(PROJECT_DIR / 'train_cnn.py'),
    '--data-dir', str(LOCAL_DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '15',
    '--batch-size', '32',
], check=True)
print('Saved results to:', OUTPUT_DIR)

## Results

Open `MyDrive/plant_classifier/outputs/cnn` in Drive. It contains the best validation checkpoint, per-class test report, confusion matrix, and training history. The test split is evaluated at the end; use validation results when changing model settings.